In [ ]:
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors

def get_descriptors_fast(formula, n_bits=1024):
    mol = Chem.MolFromSmiles(formula)
    if mol is None: return None

    # Adding hydrogens only for LabuteASA
    mol_with_h = Chem.AddHs(mol)

    # BASIC AND PEPTIDE
    descriptors = {
        'MolWt': Descriptors.MolWt(mol),
        'LogP': Descriptors.MolLogP(mol),
        'TPSA': Descriptors.TPSA(mol),
        'NumHDonors': Descriptors.NumHDonors(mol),
        'NumHAcceptors': Descriptors.NumHAcceptors(mol),
        'NumRotatableBonds': Descriptors.NumRotatableBonds(mol),
        'FractionCSP3': Descriptors.FractionCSP3(mol),
        'RingCount': Descriptors.RingCount(mol),
        'AmideBonds': rdMolDescriptors.CalcNumAmideBonds(mol),
        'ChiralCenters': len(Chem.FindMolChiralCenters(mol, includeUnassigned=True)),
        'LabuteASA': rdMolDescriptors.CalcLabuteASA(mol_with_h),
    }

    # CYCLES AND MACROCYCLES
    ri = mol.GetRingInfo()
    descriptors['Macrocycles'] = len([r for r in ri.AtomRings() if len(r) > 12])
    # Average ring size
    if descriptors['RingCount'] > 0:
        descriptors['AvgRingSize'] = sum(len(r) for r in ri.AtomRings()) / descriptors['RingCount']
    else:
        descriptors['AvgRingSize'] = 0

    # FINGERPRINTS (expand into columns)
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=n_bits)

    # An optimized way to convert to a list of numbers
    fp_array = np.zeros((0,), dtype=int)
    Chem.DataStructs.ConvertToNumpyArray(fp, fp_array)

    for i, val in enumerate(fp_array):
        descriptors[f'bit_{i}'] = val

    return descriptors



In [ ]:
path = 'Data'
folder = 'Clearance'
data_name = 'clearance'
set_name = 'test'
#func_y = 'Clearance, L/h/kg (Homo sapiens)'
func_y = 'Clearance, L/h/kg'

In [ ]:
path = 'Data'
folder = 'Vd'
data_name = 'Vd'
set_name = 'train_all_org'
#func_y = 'Clearance, L/h/kg (Homo sapiens)'
func_y = 'Vd, L/kg'

In [ ]:
path = 'Data'
folder = 'Half_life'
data_name = 'Half_life'
set_name = 'test_inside'
func_y = 'Half-life, h (Homo sapiens)'
#func_y = 'Half-life, h'

In [ ]:
hl_df = pd.read_csv(f'{path}/{folder}/{data_name}_{set_name}.csv')
#hl_df = pd.read_excel(f'{path}/{folder}/{data_name}_{set_name}.xlsx')
hl_df

,sources,Name,Canonical_smiles,Cyclic/Linear (checked),Functional_activity,PK_groups,Canonical/Non-canonical,Sequence,Small,Set,Столбец 1,Organism,"Vd, L/kg",is_Canis lupus,is_Homo sapiens,is_Macaca fascicularis,is_Mus musculus,is_Rattus norvegicus
0,chembl,CHEMBL1086218,C/C=C/C[C@@H](C)C(=O)[C@H]1C(=O)N[C@@H](C(C)C)...,Cyclic,Противоопухолевое,Группа 4: Рецепторно-сигнальные и таргетные мо...,NaN,NaN,0,Train_All_Available,NaN,Homo sapiens,1.800,False,True,False,False,False
1,chembl,CHEMBL2103735 (CETRORELIX ACETATE),CC(=O)N[C@H](Cc1ccc2ccccc2c1)C(=O)N[C@H](Cc1cc...,Linear,Гормон,Группа 2: Системные метаболические регуляторы,Non-canonical (неканонические ак),NaN,0,Train_All_Available,NaN,Homo sapiens,0.390,False,True,False,False,False
2,chembl,CHEMBL114,CC(C)(C)NC(=O)[C@@H]1C[C@@H]2CCCC[C@@H]2CN1C[C...,Linear,Saquinavir,NaN,NaN,NaN,0,Train_All_Available,NaN,Homo sapiens,3.600,False,True,False,False,False
3,chembl,CHEMBL1652593,CCC(C)CCCCC(=O)N[C@@H](CCN)C(=O)N[C@H](C(=O)N[...,Cyclic,NaN,NaN,NaN,NaN,0,Train_All_Available,NaN,Homo sapiens,0.193,False,True,False,False,False
4,chembl,CHEMBL4744444,CCCCCCCCCC(=O)N[C@@H](Cc1c[nH]c2ccccc12)C(=O)N...,Cyclic,NaN,NaN,NaN,NaN,0,Train_All_Available,NaN,Homo sapiens,0.090,False,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1660,Small_data,Flucozole,OC(Cn1cncn1)(Cn1cncn1)c1ccc(F)cc1F,NaN,NaN,NaN,NaN,NaN,1,Unused,NaN,Canis lupus,0.700,True,False,False,False,False
1661,Small_data,Fluphezine,OCCN1CCN(CCCN2c3ccccc3Sc3ccc(C(F)(F)F)cc32)CC1,NaN,NaN,NaN,NaN,NaN,1,Unused,NaN,Canis lupus,5.500,True,False,False,False,False
1662,Small_data,Butorphanol,Oc1ccc2c(c1)[C@]13CCCC[C@@]1(O)[C@H](C2)N(CC1C...,NaN,NaN,NaN,NaN,NaN,1,Unused,NaN,Canis lupus,8.400,True,False,False,False,False
1663,Small_data,Trovafloxacin,[NH3+]C1[C@H]2CN(c3nc4c(cc3F)c(=O)c(C(=O)[O-])...,NaN,NaN,NaN,NaN,NaN,1,Unused,NaN,Canis lupus,1.370,True,False,False,False,False


In [ ]:
categories = ['Homo sapiens', 'Rattus norvegicus', 'Mus musculus', 'Macaca fascicularis', 'Canis lupus']  # Порядок важен!
mapping = {cat: i for i, cat in enumerate(categories)}
hl_df['Org_ind'] = hl_df['Organism'].map(mapping)
hl_df

,sources,Name,Canonical_smiles,Cyclic/Linear (checked),Functional_activity,PK_groups,Canonical/Non-canonical,Sequence,Small,Set,Столбец 1,Organism,"Vd, L/kg",is_Canis lupus,is_Homo sapiens,is_Macaca fascicularis,is_Mus musculus,is_Rattus norvegicus,Org_ind
0,chembl,CHEMBL1086218,C/C=C/C[C@@H](C)C(=O)[C@H]1C(=O)N[C@@H](C(C)C)...,Cyclic,Противоопухолевое,Группа 4: Рецепторно-сигнальные и таргетные мо...,NaN,NaN,0,Train_All_Available,NaN,Homo sapiens,1.800,False,True,False,False,False,0
1,chembl,CHEMBL2103735 (CETRORELIX ACETATE),CC(=O)N[C@H](Cc1ccc2ccccc2c1)C(=O)N[C@H](Cc1cc...,Linear,Гормон,Группа 2: Системные метаболические регуляторы,Non-canonical (неканонические ак),NaN,0,Train_All_Available,NaN,Homo sapiens,0.390,False,True,False,False,False,0
2,chembl,CHEMBL114,CC(C)(C)NC(=O)[C@@H]1C[C@@H]2CCCC[C@@H]2CN1C[C...,Linear,Saquinavir,NaN,NaN,NaN,0,Train_All_Available,NaN,Homo sapiens,3.600,False,True,False,False,False,0
3,chembl,CHEMBL1652593,CCC(C)CCCCC(=O)N[C@@H](CCN)C(=O)N[C@H](C(=O)N[...,Cyclic,NaN,NaN,NaN,NaN,0,Train_All_Available,NaN,Homo sapiens,0.193,False,True,False,False,False,0
4,chembl,CHEMBL4744444,CCCCCCCCCC(=O)N[C@@H](Cc1c[nH]c2ccccc12)C(=O)N...,Cyclic,NaN,NaN,NaN,NaN,0,Train_All_Available,NaN,Homo sapiens,0.090,False,True,False,False,False,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1660,Small_data,Flucozole,OC(Cn1cncn1)(Cn1cncn1)c1ccc(F)cc1F,NaN,NaN,NaN,NaN,NaN,1,Unused,NaN,Canis lupus,0.700,True,False,False,False,False,4
1661,Small_data,Fluphezine,OCCN1CCN(CCCN2c3ccccc3Sc3ccc(C(F)(F)F)cc32)CC1,NaN,NaN,NaN,NaN,NaN,1,Unused,NaN,Canis lupus,5.500,True,False,False,False,False,4
1662,Small_data,Butorphanol,Oc1ccc2c(c1)[C@]13CCCC[C@@]1(O)[C@H](C2)N(CC1C...,NaN,NaN,NaN,NaN,NaN,1,Unused,NaN,Canis lupus,8.400,True,False,False,False,False,4
1663,Small_data,Trovafloxacin,[NH3+]C1[C@H]2CN(c3nc4c(cc3F)c(=O)c(C(=O)[O-])...,NaN,NaN,NaN,NaN,NaN,1,Unused,NaN,Canis lupus,1.370,True,False,False,False,False,4


In [ ]:
#hl_df['Org_ind'] = 0.0

In [ ]:

descriptors_df = pd.DataFrame()

for index, row in hl_df.iterrows():
    formula = row['Canonical_smiles']

    descriptors = get_descriptors_fast(formula)
    if descriptors:
        descriptors['Smiles'] = formula
        descriptors_df = pd.concat([descriptors_df, pd.DataFrame([descriptors])], ignore_index=True)
    else:
        print (row)
        break


In [ ]:
descriptors_df['y']= np.log10(hl_df[func_y] + 1e-6)
descriptors_df['Org_ind'] = hl_df['Org_ind']

In [ ]:
descriptors_df.to_csv(f'{path}/{folder}/{data_name}_{set_name}_rdkit.csv', index=None)
print(f'{path}/{folder}/{data_name}_{set_name}_rdkit.csv')

Data/Vd/Vd_train_all_org_rdkit.csv


In [ ]:
descriptors_df

,MolWt,LogP,TPSA,NumHDonors,NumHAcceptors,NumRotatableBonds,FractionCSP3,RingCount,AmideBonds,ChiralCenters,...,bit_1017,bit_1018,bit_1019,bit_1020,bit_1021,bit_1022,bit_1023,Smiles,y,Org_ind
0,1214.646,3.72320,275.64,4,12,15,0.777778,1,11,11,...,0,0,1,0,0,0,0,C/C=C/C[C@@H](C)C(=O)[C@H]1C(=O)N[C@@H](C(C)C)...,0.255273,0
1,1491.116,-0.41523,532.97,18,17,38,0.430556,6,13,10,...,1,0,1,0,0,0,0,CC(=O)N[C@H](Cc1ccc2ccccc2c1)C(=O)N[C@H](Cc1cc...,-0.408934,0
2,670.855,3.09240,166.75,5,7,12,0.500000,5,4,6,...,0,0,1,0,0,0,0,CC(C)(C)NC(=O)[C@@H]1C[C@@H]2CCCC[C@@H]2CN1C[C...,0.556303,0
3,1279.616,-5.54730,545.03,19,20,29,0.800000,1,11,13,...,0,0,1,1,0,0,0,CCC(C)CCCCC(=O)N[C@@H](CCN)C(=O)N[C@H](C(=O)N[...,-0.714440,0
4,1620.693,-5.62180,702.02,22,24,35,0.527778,4,14,13,...,0,0,1,0,0,0,0,CCCCCCCCCC(=O)N[C@@H](Cc1c[nH]c2ccccc12)C(=O)N...,-1.045753,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1660,306.276,0.73580,81.65,1,7,5,0.230769,3,0,0,...,0,0,0,0,0,0,0,OC(Cn1cncn1)(Cn1cncn1)c1ccc(F)cc1F,-0.154901,4
1661,437.531,4.30810,29.95,1,5,6,0.454545,4,0,0,...,0,0,0,1,0,0,0,OCCN1CCN(CCCN2c3ccccc3Sc3ccc(C(F)(F)F)cc32)CC1,0.740363,4
1662,327.468,3.36560,43.70,2,3,2,0.714286,5,0,3,...,0,0,1,0,0,0,0,Oc1ccc2c(c1)[C@]13CCCC[C@@]1(O)[C@H](C2)N(CC1C...,0.924279,4
1663,416.359,-0.15700,105.90,1,6,3,0.250000,5,0,3,...,0,0,1,0,0,0,0,[NH3+]C1[C@H]2CN(c3nc4c(cc3F)c(=O)c(C(=O)[O-])...,0.136721,4
